In [0]:
%python
# Rate source generates rows at a fixed rate — perfect for learning
# without needing Kafka or files

streaming_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

# Inspect schema — always do this first
streaming_df.printSchema()

In [0]:
# The checkpoint location is critical — it's how Spark knows
# where it left off if the stream restarts

checkpoint_path = "/Volumes/workspace/default/streaming_demo/checkpoints/rate_to_delta"
output_path = "/Volumes/workspace/default/streaming_demo/delta/raw_events"

query = (
    streaming_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(output_path)
)

# Wait for the stream to complete
query.awaitTermination()

print("Stream stopped. Rows written:", spark.read.format("delta").load(output_path).count())

In [0]:
# Restart using the SAME checkpoint — Spark resumes from where it stopped
# This proves fault tolerance / exactly-once semantics

query2 = (
    streaming_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)  # same path!
    .outputMode("append")
    .trigger(availableNow=True)
    .start(output_path)
)

query2.awaitTermination()

# Check history — notice no duplicate versions
spark.sql(f"DESCRIBE HISTORY delta.`{output_path}`").show(truncate=False)

In [0]:
# Read version 0 — the first batch ever written
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(output_path)
print("Version 0 count:", df_v0.count())

# Read current version
df_now = spark.read.format("delta").load(output_path)
print("Current count:", df_now.count())

# Time travel by timestamp
from datetime import datetime, timedelta
one_min_ago = (datetime.now() - timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
df_time = spark.read.format("delta").option("timestampAsOf", one_min_ago).load(output_path)
print("Count 1 min ago:", df_time.count())